In [ ]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.express as px
import base64
import os

from dotenv import load_dotenv

from datetime import datetime
from modules.data_loading import load_energy_community_data, load_all_metering_points_in_energy_community_data, get_postgres_engine
from sshtunnel import SSHTunnelForwarder

from modules.load_config import load_params_from_yaml

In [ ]:
# Defining logo for watermark in plots

image_filename = "../../img/icon_logo-1024x1024.png"  # local file path
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

logo = dict(
    source=f"data:image/png;base64,{encoded_image}",  # local file embedded as base64
    xref="paper", yref="paper",
    x=0.5, y=0.5,            # position: centre
    sizex=0.6, sizey=0.6,    # adjust size
    xanchor="center",
    yanchor="middle",
    opacity=0.2,             # transparency
    layer="below"            # place below the data layer
)

# import data

In [ ]:
# # PARMS
# changeable
org_id = 31

# fixed
path_to_local_data = "../../config/"
notebooks_config_file = "notebooks.yaml"

In [ ]:

config_dict = load_params_from_yaml([f"{path_to_local_data}{notebooks_config_file}"])
path_to_local_data = config_dict["PATH_TO_LOCAL_DATA"]
input_filename = config_dict["SINGLE_MPS_OF_EEG"].format(org_id=org_id)

full_filepath_to_load = f"{path_to_local_data}{input_filename}"

In [ ]:
raw_eeg = pd.read_csv(f"{full_filepath_to_load}")
raw_eeg['time'] = pd.to_datetime(raw_eeg['time'], utc=True)

print(f"{raw_eeg.dtypes}")
print(f"len: {len(raw_eeg)}")

In [ ]:
raw_eeg.describe()

# domain analysis

In [ ]:
# number of metering points

eeg_cleaned = raw_eeg.copy()
consumer = eeg_cleaned[eeg_cleaned["energy_direction"] == "C"]
generators = eeg_cleaned[eeg_cleaned["energy_direction"] == "G"]

mp_cnt = len(raw_eeg["mp_id"].unique())
cons_cnt = len(consumer["mp_id"].unique())
gen_cnt = len(generators["mp_id"].unique())

print(f"Number of all metering points in the REC: {mp_cnt}")
print(f"\t consumers: {cons_cnt} ({cons_cnt/mp_cnt*100:.1f}%)")
print(f"\t generators: {gen_cnt} ({gen_cnt/mp_cnt*100:.1f}%)")
print()
# sums

sum_eeg_cons = consumer["wt_meas_cons"].sum()
sum_eeg_gen = generators["wt_meas_gen"].sum()
sum_eeg_surp = generators["wt_surp_gen"].sum()
print(f"Total consumption of all metering points: {sum_eeg_cons:.0f} kWh")
print(f"Total production of all metering points: {sum_eeg_gen:.0f} kWh")
print(f"\tof which surplus of all metering points: {sum_eeg_surp:.0f} kWh -> {sum_eeg_surp/sum_eeg_gen*100:.0f}%")

In [ ]:
# compute the number of unique metering points per day
eeg_cleaned["time_day"] = pd.to_datetime(eeg_cleaned["time"]).dt.floor("D")

# count the number of unique org_id for each single day
daily_counts = (
    eeg_cleaned.groupby(["time_day", "energy_direction"])["mp_id"]
    .nunique()
    .reset_index(name="unique_obj_count")
)

# if energy_direction is not set, it is replaced with 0
daily_counts = (
    daily_counts
    .pivot(index="time_day", columns="energy_direction", values="unique_obj_count")
    .fillna(0)
    .reset_index()
)

In [ ]:
# plotly chart of count of metering points per day
fig = px.area(
    daily_counts,
    x="time_day",
    y=["C", "G"],  # Consumer + Generator
    title="Number of unique metering points per day (Consumer vs. Generator)",
    labels={
        "time_day": "Date",
        "value": "Number of unique metering points",
        "variable": "Energy direction",
    },
)

# fig.add_layout_image(
#     logo
# )

fig.update_layout(
    hovermode="x unified",
    legend_title_text="Energy direction",
    legend=dict(
            orientation="h",
            x=0,
            y=-0.2,
            xanchor="left",
            yanchor="top",
        ),
    template="plotly_white"
)

fig.show()


In [ ]:
# PRAMATER

# ====================================

date_start = "2025-06-18"
date_end = "2025-06-28"

# ====================================


### Total sums

In [ ]:
# compute sums per metering point for the whole time period
temp_timefiltered = eeg_cleaned[
    (eeg_cleaned["time"] > date_start) & (eeg_cleaned["time"] < date_end)
].copy()

# sum of energy consumption (wt_meas_cons) and production (wt_meas_gen) for the whole period
total_sum = (
    temp_timefiltered.groupby(["obj_id", "energy_direction"], as_index=False)[["wt_meas_gen", "wt_meas_cons"]]
    .sum()
)

# prepare for Plotly (melt)
total_sum_melted = total_sum.melt(
    id_vars=["obj_id", "energy_direction"],
    value_vars=["wt_meas_gen", "wt_meas_cons"],
    var_name="Type",
    value_name="Total"
)

total_sum_melted = total_sum_melted[total_sum_melted["Total"] != 0].copy()


In [ ]:
# plots histplot of Summed Consumption for each single mp
# "C" -> consumer / Consumer
daily_sum_c = total_sum_melted[total_sum_melted["energy_direction"] == "C"].copy()

# 📊 histogram only for energy_direction = "C"
fig = px.histogram(
    daily_sum_c,
    x="Total",
    color="Type",
    nbins=50,
    marginal="box",        # boxplot on top for an overview
    opacity=0.7,
    color_discrete_sequence=["#E66E46", "#EE715B"],
    title=f"Histogram of sums per metering point in '{date_start}' - '{date_end}'\n(consumption only - 'C')"
)

fig.add_layout_image(
    logo
)

fig.update_layout(
    template="plotly_white",
    bargap=0.05,
    legend_title_text="Measurement type",
    xaxis_title="Total consumption in kWh per metering point",
    yaxis_title="Count"
)

fig.show()


- 8 outliers

In [ ]:
sum_eeg_cons = temp_timefiltered["wt_meas_cons"].sum()
sum_eeg_comm_cov = temp_timefiltered["comm_cov"].sum()
print(f"Total consumption all MPs: {sum_eeg_cons:.2f} kWh (in period '{date_start}' - '{date_end}')")
print(f"\t of which covered by the EEG: {sum_eeg_comm_cov:.2f} kWh / {sum_eeg_comm_cov/sum_eeg_cons*100:.0f}%")

In [ ]:
# plots histplot of Summed Generation for each single mp
# "G" -> feed-in points / producers / generators
daily_sum_g = total_sum_melted[total_sum_melted["energy_direction"] == "G"].copy()

# 📊 histogram only for energy_direction = "G" (orange-yellow colour palette)
fig = px.histogram(
    daily_sum_g,
    x="Total",
    color="Type",
    nbins=50,
    marginal="box",        # boxplot on top for an overview
    opacity=0.7,
    color_discrete_sequence=["#FFD166", "#EF8A17"],  # yellow -> orange
    title=f"Histogram of total sums per metering point in '{date_start}' - '{date_end}' (generation only - 'G')"
)

fig.add_layout_image(
    logo
)

fig.update_layout(
    template="plotly_white",
    bargap=0.05,
    legend_title_text="Measurement type",
    xaxis_title="Total production in kWh per feed-in point",
    yaxis_title="Count"
)

fig.show()


- 5 outliers

In [ ]:
sum_eeg_gen = daily_sum_g["Total-Sum"].sum()
print(f"Total production all MPs: {sum_eeg_gen:.4f} kwH (in period '{date_start}' - '{date_end}')")

### Individueller Autarkiegrad

Important question/definition: what *actually* is the ratio of used energy from the EEG?

In [ ]:
# Calculation: percentage share of EEG supply per time unit
temp = temp_timefiltered[temp_timefiltered["energy_direction"]=="C"].copy()
temp["comm_cov_ratio_of_records"] = (temp["comm_cov"] / temp["wt_meas_cons"]).replace(np.nan, 1)

temp[temp["comm_cov_ratio_of_records"]>1][["time", "wt_meas_cons", "comm_cov", "comm_pot", "comm_cov_ratio_of_records"]].head()
temp[temp["comm_cov_ratio_of_records"]<1][["time", "wt_meas_cons", "comm_cov", "comm_pot", "comm_cov_ratio_of_records"]].head()

In [ ]:
result = (
    temp
    .groupby(["obj_id", "energy_direction"], as_index=False)
    .agg(
        comm_cov_ratio_of_records_mean =("comm_cov_ratio_of_records", "mean"),
        wt_meas_cons_sum=("wt_meas_cons", "sum"),
        comm_cov_sum=("comm_cov", "sum"),
        comm_pot_sum=("comm_pot", "sum"),
        
    )
)
result["comm_cov_ratio_of_sums"] = (result["comm_cov_sum"] / result["wt_meas_cons_sum"])
result.sort_values(by="comm_cov_ratio_of_records_mean").head()

In [ ]:
grp_over_time = (
    temp
    .groupby(["time", "energy_direction"], as_index=False)
    .agg(
        comm_cov_ratio_of_records_mean =("comm_cov_ratio_of_records", "mean"),
        
    )
)
grp_over_time.sort_values(by="comm_cov_ratio_of_records_mean").head()

In [ ]:
temp[["time", "obj_id", "comm_cov_ratio_of_records", "wt_meas_cons", "comm_cov"]]

In [ ]:
# len(result[result["comm_ratio_mean"] > result["comm_ratio"]])
# len(result[result["comm_ratio_mean"] < result["comm_ratio"]])

In [ ]:
# plots relationship between total energy consumptiona and ratio of community usage
fig = px.scatter(
    result,
    x="comm_ratio",
    y="wt_meas_cons_sum",
    color="energy_direction",      # optional, for colour separation by category
    hover_data=["obj_id"],         # optional, for extra info on hover
    title="Scatterplot: comm_ratio vs wt_meas_cons_sum"
)

fig.add_layout_image(
    logo
)

fig.show()


In [ ]:
# plots histplot of ratio of total community coverage / total energy consumption

# prepare data for histplot
ratio_mean_melted = result.melt(
    id_vars=["obj_id", "energy_direction"],
    value_vars=["comm_cov_ratio_of_sums"],
    var_name="Type",
    value_name="aggregated_comm_ratio"
)
ratio_aggregated_melted = ratio_mean_melted[ratio_mean_melted["aggregated_comm_ratio"] != 0].copy()

# 📊 histogram only for energy_direction = "G" (orange-yellow colour palette)
fig = px.histogram(
    ratio_mean_melted,
    x="aggregated_comm_ratio",
    color="Type",
    nbins=50,
    marginal="box",        # boxplot on top for an overview
    opacity=0.7,
    color_discrete_sequence=["#028D0E", "#006405"],  # yellow -> orange
    title=f"Histogram of the ratio of REC consumption to total consumption in '{date_start}' - '{date_end}'\n(consumer only - 'C')"
)

fig.add_layout_image(
    logo
)

fig.update_layout(
    template="plotly_white",
    bargap=0.05,
    legend_title_text="Measurement type",
    xaxis_title="Ratio [sum of REC-covered consumption / total energy consumption] in % per metering point",
    yaxis_title="Count"
)

fig.show()


- check that the MEAN per MP changes when the time period of the sums changes

## Daily load profiles

In [ ]:
def plot_profile_by_category(
    df,
    energy_col_name='sum_wt_meas_gen',
    agg_func_str='median',
    hue_col='weekday',
    extra_col=None,        # e.g. 'temp'
    logo=None, 
):
    if agg_func_str not in ['mean', 'median', 'sum', 'min', 'max', 'std']:
        raise ValueError(f"Unsupported aggregation function: {agg_func_str}")
    
    if hue_col not in df.columns:
        raise ValueError(f"'{hue_col}' is not a column in the DataFrame!")

    df["daytime"] = df.time.dt.strftime("%H:%M")

    # aggregate the energy data by hue_col & daytime
    temp_df = (
        df
        .groupby([hue_col, "daytime"])[energy_col_name]
        .agg(agg_func_str)
        .reset_index()
    )

    unique_cats = sorted(temp_df[hue_col].unique())
    color_list = px.colors.qualitative.Plotly
    colors = {cat: color_list[i % len(color_list)] for i, cat in enumerate(unique_cats)}

    fig = go.Figure()

    # plot the main data (hue_col)
    for cat in unique_cats:
        cat_df = temp_df[temp_df[hue_col] == cat]
        fig.add_trace(go.Scatter(
            x=cat_df['daytime'],
            y=cat_df[energy_col_name],
            mode='lines',
            name=f'{cat} ({agg_func_str})',
            line=dict(color=colors[cat], dash='solid'),
            yaxis='y'
        ))

    # an additional column (if given)
    if extra_col:
        if extra_col not in df.columns:
            raise ValueError(f"'{extra_col}' is not a column in the DataFrame!")

        extra_df = (
            df
            .groupby("daytime")[extra_col]
            .agg(agg_func_str)
            .reset_index()
        )

        # trace (hidden by default, but the axis stays visible!)
        fig.add_trace(go.Scatter(
            x=extra_df['daytime'],
            y=extra_df[extra_col],
            mode='lines',
            name=f'{extra_col} ({agg_func_str})',
            line=dict(color='black', dash='dot'),
            
            yaxis='y2'
        ))

        # make the y2 axis visible (with title)
        fig.update_layout(
            yaxis2=dict(
                title=extra_col,
                overlaying='y',
                side='right',
                showgrid=False,
                visible=True  # <<< HERE: visible, ALWAYS!
            )
        )

    if logo is not None:
        fig.add_layout_image(logo)


    # General layout
    fig.update_layout(
        title=f'Daily profiles by category: {hue_col} ({agg_func_str})',
        xaxis=dict(
            title='Time of day',
            tickangle=45,
            automargin=True,
            tickfont=dict(size=12)
        ),
        yaxis=dict(title=f'{energy_col_name} ({agg_func_str})'),
        legend=dict(x=0.5, y=1.15, orientation='h', xanchor='center'),
        margin=dict(b=80, t=80, l=60, r=80),
        height=600
    )

    fig.show()


In [ ]:
obj_ids_to_filter = [261, 1212]

temp = temp_timefiltered[temp_timefiltered["energy_direction"] == "C"]
temp = temp[temp["obj_id"].isin(obj_ids_to_filter)]
plot_profile_by_category(temp, energy_col_name='wt_meas_cons', agg_func_str='median', hue_col="obj_id", logo=logo)

In [ ]:
x = temp_timefiltered[temp_timefiltered["time"] == "2025-06-18 00:15:00+00:00"]

In [ ]:
temp_timefiltered.columns

In [ ]:
temp_timefiltered

In [ ]:
xx = temp_timefiltered.groupby(by="time")[['wt_meas_cons',
       'comm_pot', 'comm_cov', 'wt_meas_gen', 'wt_surp_gen']].sum().reset_index()
xx["has_surp"] = xx["wt_surp_gen"] > 0
xx[xx["has_surp"] == False]

In [ ]:
xxx = pd.merge(left=xx, right=temp_timefiltered[temp_timefiltered["energy_direction"] == "C"].copy()[["time", "wt_meas_cons", "comm_cov"]], on="time")
xxx[(xxx["has_surp"]) & (xxx["comm_cov"] > xxx["wt_meas_cons"])]

In [ ]:
temp_timefiltered[(temp_timefiltered["time"] == "2025-06-18 20:00:00+00:00") & (temp_timefiltered["wt_meas_cons"] < temp_timefiltered["comm_pot"])]

**Insight**: 
the REC internally distributes comm_cov so that every metering point, at every time step t (every quarter-hour), receives the same **share** of kWh from the REC relative to its own current grid draw. That means metering points that consume more kWh in absolute terms also draw more kWh in absolute terms from the REC's comm_cov.